# Pydantic

- can work with data in oop manner
- fields and methods
- validate data
- output json data

pydantic model created by inheriting from BaseModel

# Start without pydantic

In [5]:
class Person:
    def __init__(self, name, gender, age):
        self.name = name
        self.gender = gender
        self.age = age

person1 = Person(name="Pontus", age=31, gender="M")
person1

In [ ]:
# attribute in person1 instance
person1.name, person1.gender, person1.age

('Pontus', 'M', 31)

In [ ]:
# detta ska inte gå, därför behöver vi validering

# person2 har name=int, gender=bool, age=string, för att vi inte har hårdtypat attributerna
person2 = Person(name=321+3, gender=True, age="trehundra")
person2.name, person2.gender, person2.age

(324, True, 'trehundra')

what if we type hint?

In [14]:
class Person:
    def __init__(self, name: str, gender: str, age: int):
        self.name = name
        self.gender = gender
        self.age = age

person3 = Person("Gauss", "M", 70)
person3

type hinting är bara en "hint", du kan fortfarande ge vilka datatyper du vill, men det kan skapa logiska buggar senare vid användning

In [13]:
person4 = Person(3.1415, 2.714, "fisk")
person4.age

'fisk'

## validate our Person class

In [ ]:
class Person:
    # manuell validering, klassen blir mycket längre bara för att göra basic validering
    def __init__(self, name: str, gender: str, age: int):
        if not isinstance(name, str):
            raise TypeError(f"'name' must be of type str not {type(name)}")
        self.name = name
        
        self.gender = gender
        self.age = age

try:
    Person(name=234, gender=3, age="hej")
except TypeError as err:
    print(err)

'name' must be of type str not <class 'int'>


In [26]:
class Person:
    # manuell validering, klassen blir mycket längre bara för att göra basic validering
    def __init__(self, name: str, gender: str, age: int):
        if not isinstance(name, str):
            raise TypeError(f"'name' must be of type str not {type(name)}")
        self.name = name
        
        self.gender = gender
        self.age = age
    
    # getter
    @property
    def age(self):
        return self._age
    
    # setter, om value klarar alla valideringar, så sätter vi value till age
    @age.setter
    def age(self, value: int):
        if not isinstance(value, int):
            raise TypeError(f"age must be of type int not {type(value)} that you have provided")
        if value < 0 or value > 125:
            raise ValueError(f"age must be between 0 and 125, not {value} that you provided")
        
        self._age = value

person5 = Person(name="Bella", gender="F", age=3)
person5.age

3

In [23]:
try:
    Person(name="Bella", gender="F", age=-3)
except ValueError as err:
    print(err)

age must be between 0 and 125, not -3 that you provided


In [27]:
try:
    Person(name="Bella", gender="F", age="3")
except TypeError as err:
    print(err)

age must be of type int not <class 'str'> that you have provided


## work smarter with pydantic

In [28]:
from pydantic import BaseModel

# inherits from BaseModel -> makes it into pydantic model, but still a normal python class
class Person(BaseModel):
    name: str
    gender: str
    age: int

person6 = Person(name="Magge", gender="M", age=35)
person6

Person(name='Magge', gender='M', age=35)

In [29]:
person6.name, person6.gender, person6.age

('Magge', 'M', 35)

In [30]:
person6.age = 36
person6.age

36

In [31]:
person6

Person(name='Magge', gender='M', age=36)

In [32]:
from pydantic import ValidationError
try:
    Person(name=3.24, gender=3.525, age=-5)
except ValidationError as err:
    print(err)

2 validation errors for Person
name
  Input should be a valid string [type=string_type, input_value=3.24, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
gender
  Input should be a valid string [type=string_type, input_value=3.525, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


In [33]:
try:
    Person(name="Sofia", gender=3.525, age=-5)
except ValidationError as err:
    print(err)

1 validation error for Person
gender
  Input should be a valid string [type=string_type, input_value=3.525, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


vill fortfarande ha age som en positiv int 

In [35]:
try:
    person7 = Person(name="Sofia", gender="f", age=-5)
    print(person7)
except ValidationError as err:
    print(err)

name='Sofia' gender='f' age=-5


## Fix Person class with age validation

In [ ]:
from pydantic import Field
# Field möjliggör att konfigurera och ge extra information för en attribute/property
from typing import Literal

class Person(BaseModel):
    name: str
    gender: Literal["M", "F"]
    age: int = Field(gt=-1, lt=126)     # Field för att säga inom vilken range, gt(greater than), lt(less than)

try:
    person7 = Person(name="Sofia", gender="F", age=26)
    print(person7)
except ValidationError as err:
    print(err)

name='Sofia' gender='F' age=26


få ut person objekt som dictionary

In [39]:
person7.model_dump()

{'name': 'Sofia', 'gender': 'F', 'age': 26}

få ut person objekt som json string (serialization)     
kan skickas med API via get-request

https://pokeapi.co/api/v2/pokemon?limit=20 - url + endpoint + limit

In [ ]:
person7.model_dump_json()

'{"name":"Sofia","gender":"F","age":26}'